In [1]:
import pandas as pd
import numpy as np
import pyodbc as odbc

Reading CSV

In [2]:
df = pd.read_csv("../01-data/karachi_ecommerce_orders_dirty.csv")
print(f"Total Rows: ", len(df))

Total Rows:  378


# TRANSFORMATION PHASE

dropping duplicates

In [3]:
initial_rows = len(df)
df = df.drop_duplicates()
final_rows = len(df)
print(f"Total duplicate rows removed: ", initial_rows - final_rows)

Total duplicate rows removed:  5


Data Types & Text Standardization

In [4]:
text_columns = ['city', 'product_category', 'product_name', 'payment_method', 'order_status', 'sales_channel']

for column in text_columns:
    df[column] = df[column].astype(str).str.strip().str.title()

order_date Column Dates Format Standardization

In [5]:
#og dates ko preserve kr lia pehle
original_dates = df['order_date'].copy()

#unko date mn parse krdia
df['order_date'] = pd.to_datetime(
    original_dates,
    format='%m/%d/%Y',
    errors='coerce'
)
#jo dates fail hogyin parse hone se
mask = df['order_date'].isna()

df.loc[mask, 'order_date'] = pd.to_datetime(
    original_dates[mask],
    format='%d/%m/%Y',
    errors='coerce'
)
##dubara se wo dates identify kri jo parse na hoskin eg 2026-08-30
dash_mask = df['order_date'].isna()

df.loc[dash_mask, 'order_date'] = pd.to_datetime(
    original_dates[dash_mask],
    format='%Y-%m-%d',
    errors='coerce'
)
#dubara se wo dates identify kri jo parse na hoskin like 20260920
mask_no_slash = df['order_date'].isna()

df.loc[mask_no_slash, 'order_date'] = pd.to_datetime(
    original_dates[mask_no_slash],
    format='%Y%m%d',
    errors='coerce'
)
#wo rows lelin jahan date null hai. (unko rejected_records_logs.csv file mn save krunga, aur end mn sab variable ko ek single all_error_rows mn pd.concat krdunga then file mn write)
null_date_rows = df[df['order_date'].isna()].copy()
null_date_rows['error_reason'] = 'Missing Order Date'

df = df[df['order_date'].notna()]

#ab string format mn parse kr dia dates ko
df['order_date'] = df['order_date'].dt.strftime('%Y-%m-%d')

## kaam ki chiz ⭐⭐⭐
1. Pehle dhoondhein jahan date null hai aur wajah likhein
null_date_rows = df[df['order_date'].isna()].copy()
null_date_rows['Error_Reason'] = 'Missing Order Date'

2. Phir dhoondhein jahan Customer ID null hai aur wajah likhein
null_cust_rows = df[df['customer_id'].isna()].copy()
null_cust_rows['Error_Reason'] = 'Missing Customer ID'

3. Phir dhoondhein jahan Amount null hai aur wajah likhein
null_amount_rows = df[df['amount'].isna()].copy()
null_amount_rows['Error_Reason'] = 'Missing Amount'

4. PRO-TIP: Saare kharab variables ko EK SINGLE DataFrame mein jor (combine) dein
all_errors_df = pd.concat([null_date_rows, null_cust_rows, null_amount_rows])

5. Pipeline ke end par sirf is EK file ko write karein!
all_errors_df.to_csv('master_etl_error_log.csv', index=False)


Quantity Column Fix

In [6]:
#pehle quantity column ko numeric mn convert kr lia.
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')

#then negative quantities ko positive bana dia
df['quantity'] = df['quantity'].abs() 

#error file mn 0 quantity wali rows daal dunga
zero_qtty_rows = df[df['quantity'] == 0 ].copy()
zero_qtty_rows['error_reason'] = 'Invalid Quantity (Zero)'

#null quantity rows ko bhi file mn chipka dunga
null_qtty_rows = df[df['quantity'].isna()].copy()
null_qtty_rows['error_reason'] = 'Null Quantity Found'

df = df[(df['quantity'] > 0) & (df['quantity'].notna())]

unit_price_pkr Column Fix

In [7]:
df['unit_price_pkr'] = pd.to_numeric(df['unit_price_pkr'], errors='coerce')

#neg prices ko positive kr dia
df['unit_price_pkr'] = df['unit_price_pkr'].abs()

#zero price aur null price rows ko file mn daalna
zero_unit_price_rows = df[df['unit_price_pkr'] == 0].copy()
zero_unit_price_rows['error_reason'] = 'Zero Unit Price'

null_unit_price_rows = df[df['unit_price_pkr'].isna()].copy()
null_unit_price_rows['error_reason'] = 'Null Unit Price'

#finally zero aur null price wali rows ko df se remove
df = df[(df['unit_price_pkr'] > 0) & (df['unit_price_pkr'].notna())]

discount_pct Column Fix

In [8]:
df['discount_pct'] = pd.to_numeric(df['discount_pct'], errors='coerce')

df['discount_pct'] = df['discount_pct'].abs()

#filtering out rows that are null (btw my file has no nulls in column discount_pct)
null_discount_rows = df[df['discount_pct'].isna()].copy()
null_discount_rows['error_reason'] = 'Null Discount'

# Agar data % mein hai (e.g. 105%) to > 100, aur agar decimals mein hai to > 1.0 check karein
invalid_discount_rows = df[df['discount_pct'] > 1.0].copy()
invalid_discount_rows['error_reason'] = 'Discount Greater Than 100%'

df = df[(df['discount_pct'] >= 0.0) & (df['discount_pct'] <= 1.0) & (df['discount_pct'].notna())]

delivery_days Column Fix

In [9]:
df['delivery_days'] = pd.to_numeric(df['delivery_days'], errors='coerce')

#sirf negative days ko handle krna hai isme
df['delivery_days'] = df['delivery_days'].abs()

#Ab float se integer ('Int64') mein convert karein
df['delivery_days'] = df['delivery_days'].astype('Int64')

order_line_id column duplicates fix

In [10]:
#Duplicated Primary Key rows ko Identify aur Capture karo (for rejection log)
duplicate_pk_rows = df[df.duplicated(subset=['order_line_id'], keep='first')].copy()

#saving into rejected_records_logs.csv
duplicate_pk_rows['error_reason'] = 'Duplicate Order Line ID (PK Conflict)'

#DataFrame se duplicate order_line_ids ko drop karke pehli row retain karo
df = df.drop_duplicates(subset=['order_line_id'], keep='first')

#### customer_id Column Fix
Direct reject karne se pehle dekho ke kya us same order_id ki kisi doosri line item mein customer_id maujood hai?

- Example: Agar order_id 8003 ki line 1 mein customer_id = C1076 hai aur line 2 mein empty hai, toh iska matlab yeh same customer ka order hai! Tum isey same order_id ke group se fill (impute) kar sakte ho.

In [11]:
#har group k missing rows ko ffill aur bfill se fill krdia
df['customer_id'] = df.groupby('order_id')['customer_id'].ffill().bfill()

#wo customers jinki customer_id fill na hoski (rejected_records_logs.csv mn daal dengy)
null_customer_id_rows = df[df['customer_id'].isna()].copy()
null_customer_id_rows['error_reason'] = 'Missing Customer Id'

#sirf wohi rows rkho jahan customer_id null na ho
df = df[df['customer_id'].notna()]

city Column Fix (same as customer_id)

In [12]:
df['city'] = df.groupby('order_id')['city'].ffill().bfill()

null_city_rows = df[df['city'].isna()].copy()
null_city_rows['error_reason'] = 'Missing City'

df = df[df['city'].notna()]

payment_method column fix

In [13]:
df['payment_method'] = df.groupby('order_id')['payment_method'].ffill().bfill()

null_payment_method_rows = df[df['payment_method'].isna()].copy()
null_payment_method_rows['error_reason'] = 'Missing Payment Method'

df = df[df['payment_method'].notna()]

 product_category Column Fix
- Logic: Agar kisi row mein product_name "Yoga Mat" hai par product_category empty hai, toh dataset mein doosri rows se jahan "Yoga Mat" ki category "Sports" mili hai, wahan se fill ho sakta hai!

In [14]:
df['product_category'] = df.groupby('product_name')['product_category'].ffill().bfill()

null_prod_cat_rows = df[df['product_category'].isna()].copy()
null_prod_cat_rows['error_reason'] = 'Missing Product Category'

df = df[df['product_category'].notna()]

product_name Column Fix (isko recover nhi kia jaskta)

In [15]:
null_prod_name_rows = df[df['product_name'].isna()].copy()
null_prod_name_rows['error_reason'] = 'Missing Product Name'

df = df[df['product_name'].notna()]
print("Congratulations! Your Data is Clean.\nNow, you can insert it into SQL Server.")

Congratulations! Your Data is Clean.
Now, you can insert it into SQL Server.


## Saving cleaned data and Rejected data into a separate file

In [ ]:
#Clean data ko csv mn save dia
df.to_csv("../01-data/02-processed/01-raw/karachi_ecommerce_orders_cleaned.csv" , index=False )
print(f"Successfully inserted data into karachi_ecommerce_orders_cleaned.csv")

rows = df.shape[0]
columns = df.shape[1]
print(f"No of rows inserted: {rows}")
print(f"No of columns inserted: {columns}")

Successfully inserted data into karachi_ecommerce_orders_cleaned.csv
No of rows inserted: 360
No of columns inserted: 14


In [17]:
#Pehle sab variables ko concat krdia
rejected_data = pd.concat(
    [null_date_rows, 
    zero_qtty_rows, 
    null_qtty_rows, 
    zero_unit_price_rows,
    null_unit_price_rows,
    null_discount_rows,
    invalid_discount_rows,
    null_customer_id_rows, 
    null_city_rows,
    null_payment_method_rows,
    duplicate_pk_rows])
#Rejected data ko rejected_records_logs.csv mn daal dia
rejected_data.to_csv("../01-data/03-rejected/rejected_records_logs.csv")

print(f"Successfully inserted data into rejected_records_logs.csv")

rows = rejected_data.shape[0]
columns = rejected_data.shape[1]

print(f"No of rows inserted: {rows}")
print(f"No of columns inserted: {columns}")

Successfully inserted data into rejected_records_logs.csv
No of rows inserted: 12
No of columns inserted: 15


### use before inserting
- Insert karne se pehle NaN values ko Python None mein convert kar lo
- df_sql = df.where(pd.notnull(df), None)

- Ab tuples ki list banao
- records_to_insert = [tuple(x) for x in df_sql.to_numpy()]

## Making Data Ready for Insertion

In [18]:
#Pehle Object type mein cast karo, phir saare NaNs / pd.NA ko None (SQL NULL) banao
df_sql = df.astype(object).where(pd.notnull(df), None)

data_to_insert = []
for row in df_sql.to_numpy():
    data_to_insert.append(tuple(row))

## SQL Insertion

In [19]:
SERVER_NAME = 'HAIDERs-LAPTOP'
DATABASE_NAME = 'Karachi_ecommerce'

connection_string = (
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={SERVER_NAME};"
    f"DATABASE={DATABASE_NAME};"
    f"Trusted_connection=yes;"
)
try:
    conn = odbc.connect(connection_string)
    cursor = conn.cursor()
    #parameterized query
    insert_query = """
        INSERT INTO orders(order_id,order_line_id,order_date,customer_id,city,product_category,product_name,quantity,unit_price_pkr,discount_pct,payment_method,order_status,delivery_days,sales_channel)
        VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?,?)
    """
    #cursor.execute("TRUNCATE TABLE orders")
    cursor.executemany(insert_query, data_to_insert)

    conn.commit()

    cursor.close()
    conn.close()
    print(f"Your Data has been successfully inserted into Sql")
except Exception as e:
    print("Oops! An error Occured: ", e)

Your Data has been successfully inserted into Sql
